In [21]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [22]:
df = pd.read_csv('../data/logistics_dataset.csv')

- This dataset not given the root of how the location and zone interconnected, therefore we investigate the location and zone by using the following step :
1. Zone vs Location number : Plot the scatter plot of location and zone to see the distribution
2. Zone vs Category :Pivot the data to check the dominant of each product category in each zone

In [23]:
df_map = df.copy()

In [24]:
df_map.groupby(['zone_number']).agg({'category':'count'})

KeyError: 'zone_number'

In [ ]:
df_map.pivot_table(index='zone_number',columns='category',aggfunc='count')

KPI_score                                         daily_demand  \
category      Apparel Automotive Electronics Groceries Pharma      Apparel   
zone_number                                                                  
1                 168        185         169       141    149          168   
2                 158        142         141       172    171          158   
3                 155        155         158       141    166          155   
4                 136        176         183       164    174          136   

                                                     ... unit_price  \
category    Automotive Electronics Groceries Pharma  ...    Apparel   
zone_number                                          ...              
1                  185         169       141    149  ...        168   
2                  142         141       172    171  ...        158   
3                  155         158       141    166  ...        155   
4                  176         183       164    174  ...        136   

                                                       zone             \
category    Automotive Electronics Groceries Pharma Apparel Automotive   
zone_number                                                              
1                  185         169       141    149     168        185   
2                  142         141       172    171     158        142   
3                  155         158       141    166     155        155   
4                  176         183       164    174     136        176   

                                          
category    Electronics Groceries Pharma  
zone_number                               
1                   169       141    149  
2                   141       172    171  
3                   158       141    166  
4                   183       164    174  

[4 rows x 115 columns]

- From my finding, the location and zone are not interconnected, therefore this will be the pain point for product routing optimization, assume that this location has not been organize the zone well. 
Therefore, I will use the part of number location id as x-coordinate and running number from 0 as y-coordinate. Where all four zone colours will scatter uniformly across the full x range (1–100) at every y level.

In [28]:
# Assign x and y coordination
# Top Row (aligned, equal width)
# "Zone A Top":   (20, 30, 35, 55),
# "Zone B Top":   (40, 50, 35, 55),
# "Zone C Top":   (60, 70, 35, 55),
# "Zone D Top":   (80, 90, 35, 55),

# # Depot (keep left, centered vertically with bottom row)
# "Depot (Black)":(5, 15, 5, 25),

# # Bottom Row (aligned with top)
# "Zone A Bot":   (20, 30, 5, 25),
# "Zone B Bot":   (40, 50, 5, 25),
# "Zone C Bot":   (60, 70, 5, 25),
# "Zone D Bot":   (80, 90, 5, 25),

df['location_num'] = df['storage_location_id'].str.extract('(\d+)').astype(int)

def assign_x_y(row):
    # Zone A Top (1-50)
    if (row['zone'] == 'A') & (row['location_num'] <= 50):
        return np.random.randint(20, 30), np.random.randint(35, 55)
    # Zone A Bottom (51-100)
    elif (row['zone'] == 'A') & (row['location_num'] > 50):
        return np.random.randint(20, 30), np.random.randint(5, 25)
    # Zone B Top (1-50)
    elif (row['zone'] == 'B') & (row['location_num'] <= 50):
        return np.random.randint(40, 50), np.random.randint(35, 55)
    # Zone B Bottom (51-100)
    elif (row['zone'] == 'B') & (row['location_num'] > 50):
        return np.random.randint(40, 50), np.random.randint(5, 25)
    # Zone C Top (1-50)
    elif (row['zone'] == 'C') & (row['location_num'] <= 50):
        return np.random.randint(60, 70), np.random.randint(35, 55)
    # Zone C Bottom (51-100)
    elif (row['zone'] == 'C') & (row['location_num'] > 50):
        return np.random.randint(60, 70), np.random.randint(5, 25)
    # Zone D Top (1-50)
    elif (row['zone'] == 'D') & (row['location_num'] <= 50):
        return np.random.randint(80, 90), np.random.randint(35, 55)
    # Zone D Bottom (51-100)
    elif (row['zone'] == 'D') & (row['location_num'] > 50):
        return np.random.randint(80, 90), np.random.randint(5, 25)
    else:
        return np.nan, np.nan


for idx, row in df.iterrows():
    x, y = assign_x_y(row)
    df.at[idx, 'x'] = x
    df.at[idx, 'y'] = y

<>:17: SyntaxWarning: invalid escape sequence '\d'
<>:17: SyntaxWarning: invalid escape sequence '\d'
C:\Users\billy\AppData\Local\Temp\ipykernel_51508\626201318.py:17: SyntaxWarning: invalid escape sequence '\d'
  df['location_num'] = df['storage_location_id'].str.extract('(\d+)').astype(int)


### Step 2 : Prioritize the demanding of each item using RFM analysis

In [32]:
# R : the older restock date mean the lower recency

df['last_restock_date'] = pd.to_datetime(df['last_restock_date'])

def date_condition(x):
    if x <= np.quantile(df['last_restock_date'],q=0.25) : 
        return 1
    elif x <= np.quantile(df['last_restock_date'],q=0.5) :
        return 2
    elif x <= np.quantile(df['last_restock_date'],q=0.75):
        return 3
    else:
        return 4

df['R'] = df['last_restock_date'].apply(date_condition)

In [33]:
# F : the lower interval of days mean the more frequent of ordering



def frequency_condition(x):
    if x <= np.quantile(df['turnover_ratio'],q=0.25) : 
        return 1 # highest order since smallest inverval gaps --> most frequent
    elif x <= np.quantile(df['turnover_ratio'],q=0.5) :
        return 2
    elif x <= np.quantile(df['turnover_ratio'],q=0.75):
        return 3
    else:
        return 4

df['F'] = df['turnover_ratio'].apply(frequency_condition)

In [34]:
# M : Monetary the higher value creation , the higher order

df['Monthly_Valuation'] = (df['unit_price'] * df['total_orders_last_month']) + (df['handling_cost_per_unit'] * df['total_orders_last_month']) + (df['holding_cost_per_unit_day'] *df['stock_level']*30)


def monetary_condition(x):
    if x <= np.quantile(df['Monthly_Valuation'],q=0.25) : 
        return 1 # highest order since smallest inverval gaps --> most frequent
    elif x <= np.quantile(df['Monthly_Valuation'],q=0.5) :
        return 2
    elif x <= np.quantile(df['Monthly_Valuation'],q=0.75):
        return 3
    else:
        return 4

df['M'] = df['Monthly_Valuation'].apply(monetary_condition)

In [35]:
df['RFM_ID'] = (df['R'].astype(str) + 
                df['F'].astype(str) + 
                df['M'].astype(str))

# 2. Define the Mapping Logic
def classify_inventory(row):
    f = int(row['F'])
    m = int(row['M'])
    
    # Tier 1: The "Critical" Items (High Frequency AND High Value)
    if f >= 3 and m >= 3:
        return 'Tier 1 (Critical)'
    
    # Tier 2: The "High Value" Items (Expensive, but slow-moving)
    elif m == 4:
        return 'Tier 2 (High Value)'
    
    # Tier 3: The "High Volume" Items (Fast-moving, but low unit value)
    elif f == 4:
        return 'Tier 3 (High Volume)'
    
    # Tier 4: The "Laggards" (Low frequency and low value)
    else:
        return 'Tier 4 (Low Priority)'

# 3. Apply the classification
df['Inventory_Tier'] = df.apply(classify_inventory, axis=1)

In [36]:
df

,item_id,category,stock_level,reorder_point,reorder_frequency_days,lead_time_days,daily_demand,demand_std_dev,item_popularity_score,storage_location_id,...,KPI_score,location_num,x,y,R,F,Monthly_Valuation,M,RFM_ID,Inventory_Tier
0,ITM10000,Pharma,283,21,4,4,49.85,1.56,0.43,L82,...,0.556,82,41.0,7.0,1,1,94665.60,4,114,Tier 2 (High Value)
1,ITM10001,Automotive,301,52,9,6,23.34,2.55,0.69,L15,...,0.723,15,29.0,54.0,3,3,144044.94,4,334,Tier 1 (Critical)
2,ITM10002,Groceries,132,60,11,8,37.69,3.15,0.62,L4,...,0.680,4,40.0,45.0,2,4,48181.98,2,242,Tier 3 (High Volume)
3,ITM10003,Automotive,346,46,13,5,33.69,2.79,0.21,L95,...,0.488,95,25.0,19.0,1,1,53259.56,2,112,Tier 4 (Low Priority)
4,ITM10004,Automotive,49,55,4,6,49.58,5.23,0.31,L36,...,0.670,36,81.0,40.0,2,2,32730.73,2,222,Tier 4 (Low Priority)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3199,ITM13199,Groceries,343,21,12,2,39.88,1.30,0.34,L43,...,0.545,43,65.0,39.0,4,4,17955.39,1,441,Tier 3 (High Volume)
3200,ITM13200,Electronics,428,43,5,7,2.68,4.25,0.91,L83,...,0.605,83,49.0,7.0,4,4,85819.56,3,443,Tier 1 (Critical)
3201,ITM13201,Groceries,415,80,14,5,49.15,5.41,0.14,L11,...,0.509,11,83.0,48.0,3,2,201828.55,4,324,Tier 2 (High Value)
3202,ITM13202,Groceries,173,84,3,9,43.39,8.47,0.69,L58,...,0.565,58,41.0,22.0,1,2,68847.15,3,123,Tier 4 (Low Priority)


### Step 3 : Initialize Reorder Point (ROP) using statistical method


### The Formula
$$ROP = (d \times L) + SS$$

* **$d$**: Average Daily Demand (Units/Day).
* **$L$**: Lead Time (The number of days between placing an order and receiving it).
* **$SS$**: Safety Stock (The "buffer" to protect against stockouts).

---

### Safety Stock (SS) Calculation
To handle variability in demand, Safety Stock is calculated as:

$$SS = Z \times \sigma_d $$

| Variable | Description | Example |
| :--- | :--- | :--- |
| **$Z$** | **Service Level Factor** | 1.65 (for 90% confidence that demand will be met) |
| **$\sigma_d$** | **Standard Deviation** | Variability of daily demand |

In [37]:
# Classify Inventory_Tier and assign corresponding value
tier_map = {'Tier 1 (Critical)': 2.33, 'Tier 2 (High Value)': 1.96, 'Tier 3 (High Volume)': 1.65, 'Tier 4 (Low Priority)': 1.28}
df['Z'] = df['Inventory_Tier'].map(tier_map)

In [38]:
df['ROP'] = np.round(df['daily_demand'] * df['lead_time_days'] + (df['Z']*df['demand_std_dev']))

In [39]:
df

,item_id,category,stock_level,reorder_point,reorder_frequency_days,lead_time_days,daily_demand,demand_std_dev,item_popularity_score,storage_location_id,...,x,y,R,F,Monthly_Valuation,M,RFM_ID,Inventory_Tier,Z,ROP
0,ITM10000,Pharma,283,21,4,4,49.85,1.56,0.43,L82,...,41.0,7.0,1,1,94665.60,4,114,Tier 2 (High Value),1.96,202.0
1,ITM10001,Automotive,301,52,9,6,23.34,2.55,0.69,L15,...,29.0,54.0,3,3,144044.94,4,334,Tier 1 (Critical),2.33,146.0
2,ITM10002,Groceries,132,60,11,8,37.69,3.15,0.62,L4,...,40.0,45.0,2,4,48181.98,2,242,Tier 3 (High Volume),1.65,307.0
3,ITM10003,Automotive,346,46,13,5,33.69,2.79,0.21,L95,...,25.0,19.0,1,1,53259.56,2,112,Tier 4 (Low Priority),1.28,172.0
4,ITM10004,Automotive,49,55,4,6,49.58,5.23,0.31,L36,...,81.0,40.0,2,2,32730.73,2,222,Tier 4 (Low Priority),1.28,304.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3199,ITM13199,Groceries,343,21,12,2,39.88,1.30,0.34,L43,...,65.0,39.0,4,4,17955.39,1,441,Tier 3 (High Volume),1.65,82.0
3200,ITM13200,Electronics,428,43,5,7,2.68,4.25,0.91,L83,...,49.0,7.0,4,4,85819.56,3,443,Tier 1 (Critical),2.33,29.0
3201,ITM13201,Groceries,415,80,14,5,49.15,5.41,0.14,L11,...,83.0,48.0,3,2,201828.55,4,324,Tier 2 (High Value),1.96,256.0
3202,ITM13202,Groceries,173,84,3,9,43.39,8.47,0.69,L58,...,41.0,22.0,1,2,68847.15,3,123,Tier 4 (Low Priority),1.28,401.0


In [40]:
df.to_csv('../data/logistic_dataset_modified.csv',index=False)

### Implementation
#### Strategy 1: Risk Mitigation (Priority-Based)
    This approach focuses on specific items (e.g.Tier 1 Critical items depends on business strategy).
    Logic: Update the ROP only for critical items or where the new calculation provides a necessary safety buffer ($ROP_{new} > ROP_{old}$).
    Advantage: Minimizes operational failure and potential stockout penalties.
    Trade-off: Increases holding costs due to higher safety stock levels.Key Metric: Reduction in monthly stockout frequency.
#### Strategy 2: Cost Reduction (Lean-Capacity)
    This approach focuses on identifying too high rop items to reduce unnecessary inventory investment.
    Logic: Update the ROP only when the new calculation is lower than the current threshold ($ROP_{new} < ROP_{old}$).
    Advantage: Directly reduces holding costs and frees up frozen capital by aligning stock levels with actual usage.
    Trade-off: Does not address items currently at risk of stockouts due to under-stocking.

Note: Implementing both strategies need to be done carefully by analyzing the trade-off and financial impact again.